# Lab Exercise: Reinforcement Learning in Gridworld
This notebook implements a complete Q-Learning agent inside a 4x4 Gridworld environment.
The agent's objective is to navigate from the start square `S` to the jackpot reward `G` (+10) while avoiding hazardous pits `X` (-10) and paying a step penalty (-1) per move.

You will see firsthand how the exploration-exploitation tradeoff decides whether the agent finds the optimal strategy or settles for a mediocre local optimum.

In [ ]:
import numpy as np
import pandas as pd

# The Gridworld Environment
GRID = [
    ["S", ".", "t", "."],
    [".", "X", "X", "."],
    [".", ".", ".", "X"],
    [".", ".", ".", "G"]
]

ROWS, COLS = 4, 4
REWARD = {"G": 10.0, "t": 2.0, "X": -10.0}
STEP_R = -1.0

ACTIONS = ["^", "v", "<", ">"]
MOVES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

def step(state, a):
    """Take action a from state and return (new_state, reward, done)"""
    r, c = state
    dr, dc = MOVES[a]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < ROWS and 0 <= nc < COLS):
        nr, nc = r, c  # Walls bounce you back
    ch = GRID[nr][nc]
    if ch in REWARD:
        return (nr, nc), REWARD[ch], True
    return (nr, nc), STEP_R, False

print("THE ENVIRONMENT")
for row in GRID:
    print("   " + " ".join(row))
print("   S=start   G=jackpot (+10)   t=small prize (+2)   X=pit (-10)   step (-1)")
print("   the agent is told NONE of this — no map, no rules, no labels")

### Helper functions for Training and Policy Visualisation

In [ ]:
def run_greedy(Q):
    """Follow the learned policy with no exploration. Returns (total reward, ending cell)."""
    s, done, guard, total, r = (0, 0), False, 0, 0.0, 0.0
    while not done and guard < 100:
        guard += 1
        s, r, done = step(s, int(np.argmax(Q[s[0], s[1]])))
        total += r
    return total, GRID[s[0]][s[1]]

def train(epsilon, episodes=3000, alpha=0.1, gamma=0.95, seed=0, track=False):
    rng = np.random.RandomState(seed)
    Q = np.zeros((ROWS, COLS, 4))
    curve, online = [], []
    for ep in range(episodes):
        s, total, done, guard = (0, 0), 0.0, False, 0
        while not done and guard < 100:
            guard += 1
            if rng.rand() < epsilon:
                a = rng.randint(4)  # EXPLORE
            else:
                a = int(np.argmax(Q[s[0], s[1]]))  # EXPLOIT
            s2, r, done = step(s, a)
            future = 0.0 if done else np.max(Q[s2[0], s2[1]])
            Q[s[0], s[1], a] += alpha * (r + gamma * future - Q[s[0], s[1], a])
            s, total = s2, total + r
        online.append(total)
        if track and ep % 250 == 0:
            curve.append((ep, run_greedy(Q)[0]))
    return Q, np.array(online), curve

def show_policy(Q):
    rows = []
    for r in range(ROWS):
        rows.append(" ".join(
            GRID[r][c] if GRID[r][c] in REWARD else ACTIONS[int(np.argmax(Q[r, c]))]
            for c in range(COLS)))
    return "\n".join("   " + row for row in rows)

### Q-Learning Training (Epsilon = 0.5)

In [ ]:
Q, online, curve = train(epsilon=0.5, track=True)

print("LEARNING PROGRESS — reward the greedy policy would earn, as training proceeds")
for ep, val in curve:
    print(f"   after {ep:>4} episodes   {val:>6.1f}")

print("\nTHE LEARNED POLICY — best action found for each square")
print(show_policy(Q))

print("\nWHAT THE AGENT BELIEVES AT THE START SQUARE")
for a, name in enumerate(ACTIONS):
    print(f"   Q[start, \'{name}\'] = {Q[0, 0, a]:>6.2f}")
print(f"   -> it picks \'{ACTIONS[int(np.argmax(Q[0, 0]))]}\', worth {Q[0, 0].max():.2f}")
print("   hand-computed optimum: jackpot path 3.21, small-prize path 0.90")

### Exploration vs Exploitation
Observe how different epsilon (curiosity rate) settings impact both learning efficiency and the final policy target.

In [ ]:
print(f"   {'epsilon':>8}  {'behaviour':<16}{'settles on':>13}{'policy R':>10}{'cost while':>12}")
print(f"   {'':>8}  {'':<16}{'':>13}{'(final)':>10}{'learning':>12}")
for eps, label in [(0.0, "never explores"), (0.05, "rarely"), (0.1, "sometimes"),
                   (0.2, "regularly"), (0.3, "often"), (0.5, "half the time"),
                   (1.0, "always random")]:
    Qe, oe, _ = train(epsilon=eps)
    total, ending = run_greedy(Qe)
    name = {"G": "JACKPOT", "t": "small prize", "X": "pit"}.get(ending, "nowhere")
    print(f"   {eps:>8.2f}  {label:<16}{name:>13}{total:>10.1f}{oe[-500:].mean():>12.2f}")

### Lab Experiments (Things to Try)

#### Experiment 1: Patience vs. Curiosity
1. Set `episodes=10000` with `epsilon=0.1` inside `train`.
2. Check if patience makes up for a lack of exploration. Can the agent escape the local small-prize optimum, or does it stay locked in?

#### Experiment 2: Value Squeezing
1. Tweak the small prize value in `REWARD` from `+2` to `+8`.
2. Observe how Q-Learning handles a highly enticing local optimum. Does the agent ever choose the jackpot path now, even at `epsilon=0.5`?

#### Experiment 3: Short-Sighted Agent
1. Set `gamma=0.5` inside `train`.
2. A lower discount rate gamma makes the agent highly short-sighted, heavily devaluing future states. Notice why the agent now prefers the quick small prize even when `epsilon=0.5`!